# Time series with JSONLDB

Store one series per file, correct observations, query timestamp ranges, and work
with pandas. Then inspect the same data directly and compact deleted rows.

**Setup:** install this repository using the [README](../README.md#install), and
select a Jupyter Python kernel with that installation. Run cells in order.
All data lives in a new temporary directory; cleanup touches only that directory.
This notebook uses fixed sample values and timestamps, so no data download is needed.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import pandas as pd
from jsonldb import FolderDB

workspace = TemporaryDirectory(prefix="jsonldb-time-series-")
root = Path(workspace.name) / "observations"
root.mkdir()  # FolderDB requires an existing root.
db = FolderDB(str(root))
t0 = datetime(2026, 1, 5, 9, 30)
t1, t2, t3 = (t0 + timedelta(minutes=i) for i in (1, 2, 3))
print("Temporary database:", root)

## 1. Write two independent series

A table name identifies the series; its datetime keys identify observations.
Two series can have the same timestamps because they live in different files.
`overwrite_*` replaces the whole table. `upsert_*` creates a missing table or
updates the supplied keys. The default precision is seconds; timezone is unspecified.

In [ ]:
db.overwrite_dict("prices.AAPL", {
    t0: {"price": 185.0, "volume": 120},
    t1: {"price": 185.5, "volume": 80},
    t2: {"price": 186.0, "volume": 90},
})
msft = pd.DataFrame(
    {"price": [400.0, 401.0, 402.0], "volume": [50, 60, 70]},
    index=pd.DatetimeIndex([t0, t1, t2]),
)
db.overwrite_df("prices.MSFT", msft)
assert set(db.get_file_list()) == {"prices.AAPL", "prices.MSFT"}
assert len(db.get_dict(["prices.AAPL"])["prices.AAPL"]) == 3
assert len(db.get_df(["prices.MSFT"])["prices.MSFT"]) == 3
print(db)

## 2. Query an inclusive range or a single observation

Results are keyed by table name. Both range endpoints are included. Equal bounds
select one key. Datetime-looking keys return as naive datetimes by default;
`auto_deserialize=False` returns stored strings.

In [ ]:
window = db.get_dict(["prices.AAPL"], lower_key=t0, upper_key=t1)["prices.AAPL"]
assert list(window) == [t0, t1]
point = db.get_dict(["prices.AAPL"], t1, t1)["prices.AAPL"]
assert point == {t1: {"price": 185.5, "volume": 80}}
text_keys = db.get_dict(["prices.AAPL"], auto_deserialize=False)["prices.AAPL"]
assert t0.isoformat() in text_keys
print(window)

## 3. Correct a record and append another observation

An upsert replaces the complete record at a key; it does not merge individual
fields. Keep the intended fields in the replacement. Use enough datetime precision
to distinguish observations: different inputs that serialize to the same timestamp
share one logical key.

In [ ]:
db.upsert_dict("prices.AAPL", {
    t1: {"price": 185.6, "volume": 85},
    t3: {"price": 186.2, "volume": 110},
})
db.upsert_df("prices.MSFT", pd.DataFrame(
    {"price": [401.2], "volume": [65]}, index=pd.DatetimeIndex([t1])
))
assert db.get_dict(["prices.AAPL"], t1, t1)["prices.AAPL"][t1]["price"] == 185.6
assert db.get_df(["prices.MSFT"])["prices.MSFT"].loc[t1, "price"] == 401.2
assert len(db.get_dict(["prices.AAPL"])["prices.AAPL"]) == 4

## 4. Analyze with pandas

JSONLDB provides storage and selection. pandas supplies resampling and analysis.
A full load follows physical file order, which can change after updates; sort the
index when chronological order matters.

In [ ]:
frame = db.get_df(["prices.AAPL"])["prices.AAPL"].sort_index()
two_minute = frame.resample("2min").agg({"price": "last", "volume": "sum"})
assert frame.index.is_unique
assert list(two_minute["volume"]) == [205, 200]
print(two_minute)

## 5. Read the file directly

Every observation is a JSON object whose single key is the timestamp and whose
value is the record. No JSONLDB API is needed to inspect a file. This example has
no metadata slot. Blank lines are tombstones; duplicate keys left after an
interrupted write resolve to the last valid physical occurrence on a full scan.
See [file format](../docs/file-format.md) before building a general-purpose reader.

In [ ]:
table = root / "prices.AAPL.jsonl"
physical_lines = table.read_text(encoding="utf-8").splitlines()
file_rows = {}
for line in physical_lines:
    if line.strip():
        file_rows.update(json.loads(line))
assert file_rows == db.get_dict(["prices.AAPL"], auto_deserialize=False)["prices.AAPL"]
print("\n".join(line for line in physical_lines if line.strip()))

## 6. Delete observations and compact

Supply both bounds to range deletion. The example removes the oldest timestamp
from each series. Deletion leaves blank space; lint reclaims it, refreshes table
statistics, and writes a report. Lint may remove damaged data, so preserve a copy
and inspect findings when using it for recovery.

In [ ]:
db.delete_file_range("prices.AAPL", t0, t0)
db.delete_file_keys("prices.MSFT", [t0])
assert t0 not in db.get_dict(["prices.AAPL"])["prices.AAPL"]
size_before = table.stat().st_size
db.lint_db()
assert table.stat().st_size < size_before
assert db.get_dbmeta()["prices.AAPL"]["count"] == 3
assert list(db.get_dict(["prices.AAPL"])["prices.AAPL"]) == [t1, t2, t3]
print((root / ".jsonldb" / "lint.log").read_text())

## Cleanup and next steps

This removes only the temporary workspace created above. To restart, run the
setup cell and subsequent cells again.

Continue with [portable datasets and metadata](02_portable_datasets.ipynb) for
microsecond precision, timezone declarations, hierarchy, copying and optional Git.
There are no transactions or coordinated concurrent writers; one process owns writes.

In [ ]:
workspace.cleanup()
assert not root.exists()
print("Temporary example data removed.")